In [1]:
import torch

In [2]:
torch.__version__

'2.5.1+cu121'

In [3]:
torch.cuda.is_available()

/home/lucifer/Documents/Tech_Manthan/myenv/lib/python3.12/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


False

In [4]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


PyTorch version: 2.5.1+cu121
CUDA available: False
GPU Name: No GPU detected


In [5]:
torch.cuda.is_available()

False

5. Develop the simplest possible neural network to fit Iris Dataset for multi-class classification. Use pytorch and sklearn. 

6. For the quickstart resource of Pytorch, the accuracy there is 50%. Your task is to boost the accuracy above 78%. Do not use transfer learning. Make sure there is no overfitting by plotting the train and test loss.

In [6]:
import pandas as pd
import numpy as np
import torch.nn as nn 

In [9]:
## AND Gate Truth Table
data = {'x_1':[0,0,1,1],'x_2':[0,1,0,1],'y':[0,0,0,1]}
df = pd.DataFrame(data)

In [10]:
df

,x_1,x_2,y
0,0,0,0
1,0,1,0
2,1,0,0
3,1,1,1


In [30]:
## Building a neural network with two input and one output unit
from torch.utils.data import DataLoader,Dataset

class CustomDataset(Dataset):
    def __init__(self,df):
        self.labels = df['y'].values.astype('float32')
        self.input_data = df[['x_1','x_2']].values.astype('float32')

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self,idx):
        x = torch.tensor(self.input_data[idx])
        y = torch.tensor(self.labels[idx])
        return x,y

In [58]:
## Defining the AND Gate Neural Model
class SingleNeuron(nn.Module):
    def __init__(self):
        super(SingleNeuron,self).__init__()
        self.layers = nn.Linear(2,1)

    def forward(self,x):
        return torch.sigmoid(self.layers(x))

In [59]:
## Creating dataset and dataloader
dataset = CustomDataset(df)
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)

In [60]:
t_f,t_l = next(iter(dataloader))
print(f'Feature: {t_f}')
print(f'Label: {t_l}')

Feature: tensor([[1., 1.]])
Label: tensor([1.])


In [61]:
next(iter(dataloader))

[tensor([[0., 1.]]), tensor([0.])]

In [62]:
## initializing the model, loss and optimizer
model = SingleNeuron()
criterion = nn.BCELoss() ## Binary Cross Entropy Loss for binary output
optimizer = torch.optim.SGD(model.parameters(),lr=0.01)

In [63]:
for epoch in range(500):
    for x_batch,y_batch in dataloader:
        y_batch = y_batch.view(-1,1)
        y_pred = model(x_batch)
        loss = criterion(y_pred,y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [64]:
with torch.no_grad():
    x = torch.tensor(df[['x_1','x_2']].values,dtype=torch.float32)
    y_true = torch.tensor(df['y'].values,dtype=torch.float32).view(-1,1)
    y_pred = model(x)
    predictions = (y_pred > 0.5).float()
    accuracy = (predictions == y_true).float().mean()

print(f"Final Loss: {loss.item():.4f}")
print(f"Accuracy: {accuracy.item() * 100:.2f}%")

Final Loss: 0.1449
Accuracy: 100.00%


In [65]:
## Saving the model
torch.save(model.state_dict(),"and_gate_model.pth")

In [66]:
# loading the model
loaded_model = SingleNeuron()
loaded_model.load_state_dict(torch.load("and_gate_model.pth"))

/tmp/ipykernel_94272/1274445603.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.load_state_dict(torch.load("and_gate_model.pth"))


<All keys matched successfully>

In [67]:
## Predicting on the dataset
loaded_model.eval()

with torch.no_grad():
    test_input = torch.tensor([[1.0, 1.0]])
    prediction = loaded_model(test_input)
    print(f"Prediction for [1,1]: {(prediction.item() > 0.5)}")

Prediction for [1,1]: True


## Trying for XOR Gate

In [68]:
xor_data = {'x_1':[0,0,1,1],'x_2':[0,1,0,1],'y':[0,1,1,0]}
df_xor = pd.DataFrame(xor_data)
dataset_xor = CustomDataset(df_xor)
xor_dataloader = DataLoader(dataset_xor,batch_size=1,shuffle=True)

In [73]:
xor_model = SingleNeuron()
optimizer_xor = torch.optim.SGD(xor_model.parameters(),lr=0.1)

In [74]:
for epoch in range(1000):
    for x,y in xor_dataloader:
        y = y.view(-1,1)
        output = xor_model(x)
        loss = criterion(output,y)
        optimizer_xor.zero_grad()
        loss.backward()
        optimizer_xor.step()

In [75]:
# Checking Accuracy
with torch.no_grad():
    inputs = torch.tensor(df_xor[['x_1','x_2']].values,dtype=torch.float32)
    targets = torch.tensor(df_xor['y'].values,dtype=torch.float32).view(-1,1)
    outputs = xor_model(inputs)
    predicted = (output > 0.5).float()
    accuracy = (predicted == targets).float().mean()

print(f"Final Accuracy for xor_gate is : {accuracy.item() * 100:2f}%")


Final Accuracy for xor_gate is : 50.000000%


For AND Gate:
- Single neuron with sigmoid activation and binary cross entropy loss is sufficient 
- Model reaches 100% accuracy within few epochs (I trained upto 500)
- Problem is linearly separable

For XOR Gate:
- Single neuron cannot learn XOR logic
- Even after 1000 epochs, accuracy doesn't reach 100%
- XOR problem is not linearly separable, so a single layer model is not sufficient for this task 

Conclusion:

- Linearly separable problems like AND can be solved using a single neuron.
- Non-linearly separable problems like XOR require a multi-layer neural network (MLP).

### What is batch, iteration and epoch?

1. Epoch

- An epoch is one complete pass through the entire dataset. For example: "Showing all training samples to the model once"

2. Batch

- A batch is a subset of the data used in one forward and backward pass. If dataset is small we may feed the whole data as single batch and if the dataset is big we don't usually feed the whole dataset at once, instead we break it into batches

3. Iteration

- An iteration is one update step of the model (forward + backward pass) using one batch. "Each batch we feed to the model leads to one iteration"